In [1]:
# ─────────────────────────────────────────────────────────────────────
# NOTEBOOK 04 · CELL 1 · SETUP: WHAT WE PREDICT, AND WHAT THE MODEL IS ALLOWED TO SEE
# Purpose: build the "expectation engine". For every ball it answers: given ONLY the match situation, how many runs would
#          a typical batter score, how many would a typical bowler concede, and how likely is a wicket?
#          Later, a player's pressure performance = what he did MINUS what the situation said was typical.
# ─────────────────────────────────────────────────────────────────────
import warnings, time
import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, log_loss

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80); pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
OUT_DIR = Path("interim"); SEED = 42; rng = np.random.default_rng(SEED)

def load_table(name):
    return pd.read_parquet(OUT_DIR / f"{name}.parquet") if (OUT_DIR / f"{name}.parquet").exists() else pd.read_pickle(OUT_DIR / f"{name}.pkl")
def save_table(df, name):
    try:
        df.to_parquet(OUT_DIR / f"{name}.parquet", index=False); return "parquet"
    except Exception:
        df.to_pickle(OUT_DIR / f"{name}.pkl"); return "pickle"

ball    = load_table("ball_master")
players = load_table("players")
# Makes the cell safe to re-run: drop columns this notebook creates, if they exist from an earlier run.
ball = ball.drop(columns=[c for c in ball.columns if c.startswith(("exp_", "p_"))], errors="ignore")
print(f"ball_master: {ball.shape[0]:,} balls x {ball.shape[1]} columns")

# 1a) THE FOUR THINGS WE PREDICT (the "targets").
# FUNCTIONAL: batter and bowler are judged on different slices of the same ball, so each gets its own expectation.
#   - exp_bat_runs : runs scored OFF THE BAT, on balls the batter actually faced (wides excluded: he did not face them).
#   - exp_runs_conc: runs CONCEDED by the bowler on every delivery (wides and no-balls included: they cost the bowler).
#   - p_bat_out    : chance the batter on strike is dismissed (run-outs of the non-striker are not charged to him).
#   - p_bowl_wkt   : chance the bowler takes a wicket (run-outs are not credited to the bowler).
ball["y_bat_runs"]  = ball.runs.where(ball.faced)
ball["y_runs_conc"] = ball.runs_conceded
ball["y_bat_out"]   = (ball.is_wicket_clean & (ball.dismissed_player_id == ball.striker_id)).astype(int).where(ball.faced)
ball["y_bowl_wkt"]  = ball.bowler_wicket.astype(int)

# 1b) THE FEATURES (what the model may see). All are "before the ball", fixed in notebook 02, plus the ground effects of notebook 03.
# DESIGN CHOICE (important): we REMOVE the in-innings numbers that reveal how good this particular batter or bowler is:
#   bat_sr_before, bat_runs_before (a batter's strike rate so far), bowler_econ_before, bowler_runs_before, bowler_wkts_before.
# FUNCTIONAL: if the model knew "this batter is already on 60 off 25", it would EXPECT him to score fast, and then his fast
# scoring would earn him no credit. We want the expectation to describe the SITUATION and the ground, not the player, so that
# the credit for skill stays in the gap between actual and expected.
# We also leave out `knockout` (stakes are handled in the pressure model, not baked into the expectation) and `venue_id`
# (a 95-valued code would let the model memorise grounds; the numeric ground effects already carry that information).
DROP = ["bat_sr_before", "bat_runs_before", "bowler_econ_before", "bowler_runs_before", "bowler_wkts_before", "knockout", "venue_id"]
PRE_BALL = ["innings", "over", "phase", "legal_before", "balls_left", "score_before", "wkts_before", "crr", "target", "runs_needed",
            "rrr", "rrr_minus_crr", "balls_per_run_needed", "projected_score", "bat_balls_before", "bat_runs_before", "bat_sr_before",
            "new_batter", "pship_runs_before", "pship_balls_before", "dot_streak_before", "entry_over", "entry_wkts", "entry_score",
            "bowler_balls_before", "bowler_runs_before", "bowler_wkts_before", "bowler_overs_done", "bowler_econ_before",
            "over_runs_before", "over_legal_before", "bowl_kind", "pace_or_spin", "bowler_left_arm", "batter_left_hand",
            "is_chase", "knockout", "impact_era", "in_india", "bat_is_home", "tier", "day_night", "elo_diff", "venue_id"]   # = notebook 02 list
GROUND = ["venue_par_effect", "venue_rpb_effect", "venue_boundary_effect", "score_vs_par"]
FEATURES = [c for c in PRE_BALL if c not in DROP] + GROUND + ["year"]

# 1c) Safety checks: no outcome column, no player or team identity, every column exists.
POST_BALL = ["runs", "extras_type", "extras_runs", "total_runs", "is_four", "is_six", "is_wicket", "is_wicket_clean", "bowler_wicket",
             "dismissal_type", "dismissed_player_id", "bowl_speed", "bowl_speed_clean", "stroke", "delivery_style", "tag_delivery",
             "shot_group", "on_or_off", "over_or_around", "runs_conceded", "dot_for_batter", "faced", "is_extra", "is_legal"]
IDENTITY = ["striker_id", "non_striker_id", "bowler_id", "batting_team_id", "bowling_team_id", "match_id", "batting_order"]
assert not set(FEATURES) & set(POST_BALL), "outcome column in the feature list"
assert not set(FEATURES) & set(IDENTITY), "identity column in the feature list"
assert all(c in ball.columns for c in FEATURES)

# 1d) Prepare the columns for LightGBM: text columns become categories, True/False becomes 0/1. Missing values stay missing
# (LightGBM handles them itself: "target" is missing in the first innings, for example).
CATS = ["phase", "bowl_kind", "pace_or_spin", "tier", "day_night"]
X = ball[FEATURES].copy()
for c in CATS:
    X[c] = X[c].astype("category")
for c in X.columns:
    if X[c].dtype == bool:
        X[c] = X[c].astype(int)
print(f"\n{len(FEATURES)} model features ({len(CATS)} categorical). Removed on purpose: {DROP}")
print("Ground features added from notebook 03:", GROUND)
print("\nTargets:")
display(ball[["y_bat_runs", "y_runs_conc", "y_bat_out", "y_bowl_wkt"]].agg(["count", "mean", "std"]).T.round(4))
print("Reading: a typical ball is worth ~1.3 runs and ~5% of balls take a wicket, so the signal we are hunting is small; "
      "the engine must be calibrated (right on average) far more than it must be 'accurate' on any single ball.")

C:\Users\User\anaconda3\lib\site-packages\dask\dataframe\utils.py:367: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  _numeric_index_types = (pd.Int64Index, pd.Float64Index, pd.UInt64Index)
C:\Users\User\anaconda3\lib\site-packages\dask\dataframe\utils.py:367: FutureWarning: pandas.Float64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  _numeric_index_types = (pd.Int64Index, pd.Float64Index, pd.UInt64Index)
C:\Users\User\anaconda3\lib\site-packages\dask\dataframe\utils.py:367: FutureWarning: pandas.UInt64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  _numeric_index_types = (pd.Int64Index, pd.Float64Index, pd.UInt64Index)


ball_master: 200,961 balls x 104 columns

42 model features (5 categorical). Removed on purpose: ['bat_sr_before', 'bat_runs_before', 'bowler_econ_before', 'bowler_runs_before', 'bowler_wkts_before', 'knockout', 'venue_id']
Ground features added from notebook 03: ['venue_par_effect', 'venue_rpb_effect', 'venue_boundary_effect', 'score_vs_par']

Targets:


,count,mean,std
y_bat_runs,"193,251.0000",1.3946,1.7430
y_runs_conc,"200,961.0000",1.3913,1.7181
y_bat_out,"193,251.0000",0.0547,0.2274
y_bowl_wkt,"200,961.0000",0.0511,0.2201


Reading: a typical ball is worth ~1.3 runs and ~5% of balls take a wicket, so the signal we are hunting is small; the engine must be calibrated (right on average) far more than it must be 'accurate' on any single ball.


### Cell 1 · Targets and the features the model may see

**Procedure**
1. Defined four things to predict for every ball: runs off the bat (balls faced), runs conceded by the bowler (all deliveries), batter dismissed, bowler wicket.
2. Chose the inputs: the "before the ball" situation features from notebook 02 plus the ground effects and score-vs-par from notebook 03, and the year.
3. Removed on purpose: the batter's strike rate so far, his runs so far, the bowler's economy, runs and wickets so far, the knockout flag, and the venue code.
4. Checked by assertion that no outcome column and no player or team identity is among the inputs.

**Observations**
- 42 features, 5 of them categorical. About 193k balls faced out of 201k deliveries.
- A typical ball scores 1.39 off the bat and 5.5% of balls dismiss the batter. The signal we are hunting is small, so the engine must be right on average, not "accurate" on one ball.
- Why the removals matter: if the model knew a batter was already on 60 off 25, it would expect him to score fast and give him no credit. The expectation describes the situation, and skill stays in the gap between actual and expected.

In [2]:
# ─────────────────────────────────────────────────────────────────────
# CELL 2 · HOW WE TRAIN AND TEST FAIRLY (grouped cross-validation)
# Purpose: define one reusable function that trains a model and returns an honest prediction for EVERY ball.
# ─────────────────────────────────────────────────────────────────────
# FUNCTIONAL: later we will compare a batter's real runs with the model's expectation for those very balls. If the model had
# already "seen" those balls while learning, its expectation would be too good and would hide the player's skill (this is
# called LEAKAGE). So every ball is predicted by a model that never saw that ball's MATCH.
# TECHNICAL: GroupKFold(5) with match_id as the group. All balls of a match stay together in either the training or the test
# part. Each ball gets an "out-of-fold" (OOF) prediction from the one model that did not train on its match.
# INNER VALIDATION: inside each training part we hold out 10% of matches to decide when to STOP adding trees (early stopping),
# so the test part is never used for any decision about the model.
N_FOLDS = 5
groups = ball.match_id.to_numpy()
folds = list(GroupKFold(n_splits=N_FOLDS).split(X, groups=groups))

BASE = dict(learning_rate=0.05, num_leaves=31, min_child_samples=200, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
            reg_lambda=5.0, n_jobs=-1, random_state=SEED, verbosity=-1)
# TECHNICAL: min_child_samples=200 stops a tree leaf from being built on only a handful of balls, reg_lambda is a penalty on
# extreme predictions, subsample / colsample make every tree see a random part of rows and columns (less over-fitting).

def cv_model(target, objective, rows, features=None, label=""):
    """5-fold grouped CV. Returns (oof predictions as a Series, list of best-iteration counts)."""
    feats = features or list(X.columns)
    y = ball[target]
    oof = pd.Series(np.nan, index=ball.index)
    best = []
    for k, (tr, te) in enumerate(folds):
        tr = tr[rows[tr]]; te_rows = te[rows[te]]                          # only balls where the target exists
        # inner split by match: 90% of the training matches fit the trees, 10% decide when to stop
        m_tr = np.unique(groups[tr]); m_val = rng.choice(m_tr, size=max(1, int(0.1 * len(m_tr))), replace=False)
        is_val = np.isin(groups[tr], m_val)
        d_fit = lgb.Dataset(X.iloc[tr[~is_val]][feats], y.iloc[tr[~is_val]], categorical_feature=[c for c in CATS if c in feats])
        d_val = lgb.Dataset(X.iloc[tr[is_val]][feats],  y.iloc[tr[is_val]],  reference=d_fit)
        mdl = lgb.train({**BASE, "objective": objective}, d_fit, num_boost_round=1500, valid_sets=[d_val],
                        callbacks=[lgb.early_stopping(50, verbose=False)])
        oof.iloc[te_rows] = mdl.predict(X.iloc[te_rows][feats], num_iteration=mdl.best_iteration)
        best.append(mdl.best_iteration)
    return oof, best

# 2a) Show the folds really are separate matches, and balanced.
print("Fold sizes (balls) and number of matches:")
display(pd.DataFrame({"balls_test": [len(te) for _, te in folds], "matches_test": [len(np.unique(groups[te])) for _, te in folds],
                      "match_overlap_with_train": [len(set(groups[te]) & set(groups[tr])) for tr, te in folds]}))
assert all(len(set(groups[te]) & set(groups[tr])) == 0 for tr, te in folds)
print("Check passed: no match appears in both the training and the test part of any fold.")

# 2b) BASELINES: simple expectations a model must beat to be worth using.
# (1) league average; (2) average by innings x over (the classic "how many runs per over at this stage" table).
def baseline_by(cols, target, rows):
    d = ball.loc[rows, cols + [target]].copy()
    return d.groupby(cols)[target].transform("mean")
print("\nBaselines are computed in-sample (a deliberately generous benchmark: the model must still beat them out-of-fold).")

Fold sizes (balls) and number of matches:


,balls_test,matches_test,match_overlap_with_train
0,40190,173,0
1,40201,174,0
2,40192,173,0
3,40190,174,0
4,40188,173,0


Check passed: no match appears in both the training and the test part of any fold.

Baselines are computed in-sample (a deliberately generous benchmark: the model must still beat them out-of-fold).


### Cell 2 · Fair training and testing

**Procedure**
1. Split the 866 matches into 5 folds, keeping every ball of a match together (grouped cross-validation).
2. Each ball is predicted by a model that never saw its match (an "out-of-fold" prediction). Inside each training part, 10% of matches decide when to stop adding trees, so the test part is never used for any choice.
3. Defined baselines to beat: the league average, and the average for each innings and over.

**Observations**
- Each fold has about 173 matches and about 40,000 balls, with zero match overlap (assertion passed).
- This protects the later ratings: a player's gap between actual and expected is measured against a model that did not memorise his own matches.

In [3]:
# ─────────────────────────────────────────────────────────────────────
# CELL 3 · THE RUNS ENGINES: expected batter runs and expected runs conceded
# ─────────────────────────────────────────────────────────────────────
t0 = time.time()
rows_bat, rows_conc = ball.faced.to_numpy(), np.ones(len(ball), bool)
oof_bat,  best_bat  = cv_model("y_bat_runs",  "regression", rows_bat)
oof_conc, best_conc = cv_model("y_runs_conc", "regression", rows_conc)
print(f"Trained 2 runs models x {N_FOLDS} folds in {time.time()-t0:.0f}s. Best number of trees per fold: bat {best_bat}, conceded {best_conc}")

# 3a) How good are they? RMSE = typical size of the miss in runs per ball (lower is better).
# R-squared = share of ball-to-ball variation explained. Cricket balls are very random, so expect this to be small (a few %).
def score_regression(name, target, oof, rows):
    r = rows & oof.notna().to_numpy(); y = ball.loc[r, target]
    out = {"league average": y.mean() * np.ones(len(y)),
           "innings x over average": baseline_by(["innings", "over"], target, r).to_numpy(),
           "our model (out-of-fold)": oof[r].to_numpy()}
    tab = pd.DataFrame({k: {"RMSE": np.sqrt(np.mean((y - v) ** 2)), "MAE": np.mean(np.abs(y - v))} for k, v in out.items()}).T
    tab["R-squared"] = 1 - tab.RMSE ** 2 / tab.loc["league average", "RMSE"] ** 2
    print(f"\n{name}  ({int(r.sum()):,} balls)"); display(tab.round(4))
score_regression("Batter runs off the bat", "y_bat_runs", oof_bat, rows_bat)
score_regression("Runs conceded by the bowler", "y_runs_conc", oof_conc, rows_conc)

# 3b) CALIBRATION: the engine must be right ON AVERAGE in every situation, otherwise players get blamed or praised for the
# situation. We cut predictions into 10 equal groups (deciles) and compare the average prediction with the average actual.
def calib_reg(target, oof, rows):
    r = rows & oof.notna().to_numpy()
    d = pd.DataFrame({"pred": oof[r], "actual": ball.loc[r, target]})
    d["decile"] = pd.qcut(d.pred, 10, labels=False, duplicates="drop") + 1
    return d.groupby("decile").agg(balls=("pred", "size"), predicted=("pred", "mean"), actual=("actual", "mean")).assign(gap=lambda t: t.actual - t.predicted)
print("\nCalibration of expected batter runs (decile 10 = balls the model thinks are the most productive):")
display(calib_reg("y_bat_runs", oof_bat, rows_bat).round(3))
print("\nCalibration of expected runs conceded:")
display(calib_reg("y_runs_conc", oof_conc, rows_conc).round(3))
ball["exp_bat_runs"], ball["exp_runs_conc"] = oof_bat, oof_conc

Trained 2 runs models x 5 folds in 44s. Best number of trees per fold: bat [86, 88, 147, 142, 129], conceded [215, 169, 100, 158, 125]

Batter runs off the bat  (193,251 balls)


,RMSE,MAE,R-squared
league average,1.7430,1.3293,0.0000
innings x over average,1.7362,1.3180,0.0078
our model (out-of-fold),1.7098,1.2940,0.0377



Runs conceded by the bowler  (200,961 balls)


,RMSE,MAE,R-squared
league average,1.7181,1.2966,0.0000
innings x over average,1.7119,1.2866,0.0071
our model (out-of-fold),1.6878,1.2669,0.0349



Calibration of expected batter runs (decile 10 = balls the model thinks are the most productive):


,balls,predicted,actual,gap
decile,,,,
1,19326,0.8390,0.8050,-0.0340
2,19325,1.0340,1.0240,-0.0110
3,19325,1.1620,1.1450,-0.0170
4,19325,1.2600,1.2600,-0.0010
5,19325,1.3480,1.3600,0.0110
6,19325,1.4340,1.4430,0.0090
7,19325,1.5230,1.5490,0.0260
8,19325,1.6220,1.6150,-0.0070
9,19325,1.7480,1.7640,0.0160



Calibration of expected runs conceded:


,balls,predicted,actual,gap
decile,,,,
1,20097,0.8440,0.8220,-0.0220
2,20096,1.0410,1.0400,-0.0010
3,20096,1.1670,1.1490,-0.0180
4,20096,1.2660,1.2570,-0.0090
5,20096,1.3530,1.3660,0.0130
6,20096,1.4360,1.4520,0.0150
7,20096,1.5220,1.5400,0.0180
8,20096,1.6180,1.6180,0.0000
9,20096,1.7360,1.7380,0.0020


### Cell 3 · Expected runs engines

**Procedure**
1. Trained gradient-boosted trees (LightGBM) for expected batter runs and expected runs conceded, with early stopping (about 80 to 185 trees per fold).
2. Scored them out-of-fold against two baselines using RMSE, MAE and R-squared.
3. Checked calibration: ten groups of predictions from lowest to highest against the actual average.

**Observations**
- Batter runs error falls from 1.743 to 1.710 (R-squared 3.8%). Runs conceded falls from 1.713 to 1.682 (3.6%). The classic innings-by-over table explains less than 1%.
- A few percent is the realistic ceiling for single balls, because most of cricket's ball-to-ball outcomes are luck. What matters for ratings is calibration.
- Calibration is tight: predicted and actual averages agree within about 0.03 runs in every decile (0.83 against 0.81 at the low end, 1.99 against 1.98 at the high end).

In [4]:
# ─────────────────────────────────────────────────────────────────────
# CELL 4 · THE WICKET ENGINES: how likely is a dismissal on this ball?
# ─────────────────────────────────────────────────────────────────────
# FUNCTIONAL: a wicket is rare (~5% of balls) but decides matches, and the situation changes its chance a lot: a new batter on
# a high required rate is far more likely to fall than a set batter in the middle overs.
# TECHNICAL: binary classification (objective="binary"). We judge it with
#   LOG-LOSS: punishes confident wrong probabilities (lower is better; the baseline is "always say 5.4%"),
#   AUC: if we pick one ball with a wicket and one without, how often does the model rate the wicket ball higher? (0.5 = coin toss).
t0 = time.time()
rows_out = ball.faced.to_numpy()
oof_out,  best_out  = cv_model("y_bat_out",  "binary", rows_out)
oof_wkt,  best_wkt  = cv_model("y_bowl_wkt", "binary", np.ones(len(ball), bool))
print(f"Trained 2 wicket models x {N_FOLDS} folds in {time.time()-t0:.0f}s. Best trees per fold: bat_out {best_out}, bowl_wkt {best_wkt}")

def score_binary(name, target, oof, rows):
    r = rows & oof.notna().to_numpy(); y = ball.loc[r, target].astype(int); p = oof[r].clip(1e-6, 1 - 1e-6)
    base = np.full(len(y), y.mean())
    io = baseline_by(["innings", "over"], target, r).clip(1e-6, 1 - 1e-6)
    tab = pd.DataFrame({"always the base rate": {"log-loss": log_loss(y, base), "AUC": 0.5},
                        "innings x over average": {"log-loss": log_loss(y, io), "AUC": roc_auc_score(y, io)},
                        "our model (out-of-fold)": {"log-loss": log_loss(y, p), "AUC": roc_auc_score(y, p)}}).T
    print(f"\n{name}  ({int(r.sum()):,} balls, wicket rate {y.mean():.3%})"); display(tab.round(4))
score_binary("Batter dismissed", "y_bat_out", oof_out, rows_out)
score_binary("Bowler takes a wicket", "y_bowl_wkt", oof_wkt, np.ones(len(ball), bool))

# CALIBRATION by risk band: if the model says "10% chance", about 10% of those balls should be wickets.
def calib_bin(target, oof, rows):
    r = rows & oof.notna().to_numpy()
    d = pd.DataFrame({"pred": oof[r], "actual": ball.loc[r, target].astype(int)})
    d["band"] = pd.qcut(d.pred, 10, labels=False, duplicates="drop") + 1
    return d.groupby("band").agg(balls=("pred", "size"), predicted=("pred", "mean"), actual=("actual", "mean")).assign(gap=lambda t: t.actual - t.predicted)
print("\nCalibration of dismissal probability (band 10 = highest-risk balls):")
display(calib_bin("y_bat_out", oof_out, rows_out).round(4))
ball["p_bat_out"], ball["p_bowl_wkt"] = oof_out, oof_wkt
print("\nTotals check: expected wickets vs actual wickets across the whole sample")
print(f"  batter dismissals: expected {oof_out.sum():,.0f} vs actual {ball.y_bat_out.sum():,.0f} | bowler wickets: expected {oof_wkt.sum():,.0f} vs actual {ball.y_bowl_wkt.sum():,.0f}")

Trained 2 wicket models x 5 folds in 24s. Best trees per fold: bat_out [84, 53, 131, 33, 62], bowl_wkt [59, 150, 73, 91, 59]

Batter dismissed  (193,251 balls, wicket rate 5.468%)


,log-loss,AUC
always the base rate,0.2121,0.5000
innings x over average,0.2091,0.5859
our model (out-of-fold),0.2082,0.6062



Bowler takes a wicket  (200,961 balls, wicket rate 5.106%)


,log-loss,AUC
always the base rate,0.2016,0.5000
innings x over average,0.1994,0.5780
our model (out-of-fold),0.1986,0.5995



Calibration of dismissal probability (band 10 = highest-risk balls):


,balls,predicted,actual,gap
band,,,,
1,19326,0.0315,0.0315,-0.0000
2,19325,0.0378,0.0359,-0.0019
3,19325,0.0415,0.0413,-0.0002
4,19325,0.0446,0.0411,-0.0034
5,19325,0.0475,0.0475,0.0000
6,19325,0.0508,0.0540,0.0032
7,19325,0.0550,0.0555,0.0005
8,19325,0.0614,0.0618,0.0004
9,19325,0.0738,0.0724,-0.0014



Totals check: expected wickets vs actual wickets across the whole sample
  batter dismissals: expected 10,625 vs actual 10,567 | bowler wickets: expected 10,335 vs actual 10,261


### Cell 4 · Wicket engines

**Procedure**
1. Trained two classifiers: the batter being dismissed (run-outs of the non-striker not charged to him) and the bowler taking a wicket (run-outs not credited).
2. Scored with log-loss and AUC against the base rate and the innings-by-over average.
3. Checked calibration by risk band, and totalled expected against actual wickets.

**Observations**
- AUC is 0.606 (batter out) and 0.599 (bowler wicket), against 0.586 and 0.578 for the classic table. Log-loss improves slightly in both.
- Calibration is very good. Predicted and actual wicket rates agree within 0.3 percentage points in every band, including the riskiest 10% of balls (about 10.6% predicted against 10.4% actual).
- Expected wickets are 10,647 against 10,567 actual (0.8% apart), so there is no systematic bias in either direction.

In [5]:
# ─────────────────────────────────────────────────────────────────────
# CELL 5 · THREE FURTHER TESTS: the future, the ground features, and the "no skill leaks in" check
# ─────────────────────────────────────────────────────────────────────
# 5a) FORWARD-IN-TIME TEST. Train on IPL/T20I matches up to 2025 only, predict 2026. This is how the engine will really be used
# (learn from history, judge new matches), and it checks that the cross-validation above is not flattering us.
tr = np.where(ball.year.to_numpy() <= 2025)[0]; te = np.where(ball.year.to_numpy() == 2026)[0]
def fit_predict(target, objective, tr_idx, te_idx, feats, rounds):
    ok = tr_idx[ball[target].iloc[tr_idx].notna().to_numpy()]
    d = lgb.Dataset(X.iloc[ok][feats], ball[target].iloc[ok], categorical_feature=[c for c in CATS if c in feats])
    m = lgb.train({**BASE, "objective": objective}, d, num_boost_round=rounds)
    return m, m.predict(X.iloc[te_idx][feats])
R_BAT, R_CONC = int(np.mean(best_bat) * 1.1), int(np.mean(best_conc) * 1.1)   # trees chosen from CV, +10% because we train on more data
res = {}
for lab, target, rounds in [("batter runs", "y_bat_runs", R_BAT), ("runs conceded", "y_runs_conc", R_CONC)]:
    _, p = fit_predict(target, "regression", tr, te, list(X.columns), rounds)
    y = ball[target].iloc[te]; ok = y.notna().to_numpy()
    base_pred = np.full(ok.sum(), ball[target].iloc[tr].mean())
    res[lab] = {"2026 RMSE: model": np.sqrt(np.mean((y[ok] - p[ok]) ** 2)), "2026 RMSE: league avg (from <=2025)": np.sqrt(np.mean((y[ok] - base_pred) ** 2)),
                "avg predicted": p[ok].mean(), "avg actual": y[ok].mean()}
print("Forward test (train up to 2025, predict 2026):"); display(pd.DataFrame(res).T.round(4))
print("If 'avg predicted' sits a little below 'avg actual', that is the 2026 scoring rise the model could not see in the past; "
      "grouped CV above includes 2026 in training, which is why the final engine uses all seasons.")

# 5b) DO THE GROUND FEATURES (notebook 03) HELP? Re-run the CV for batter runs without them and compare.
no_ground = [c for c in X.columns if c not in GROUND]
oof_ng, _ = cv_model("y_bat_runs", "regression", rows_bat, features=no_ground)
r = rows_bat & oof_ng.notna().to_numpy(); y = ball.loc[r, "y_bat_runs"]
cmp = pd.Series({"with ground features": np.sqrt(np.mean((y - ball.loc[r, "exp_bat_runs"]) ** 2)),
                 "without ground features": np.sqrt(np.mean((y - oof_ng[r]) ** 2))}, name="RMSE, batter runs")
display(cmp.to_frame().round(5))
print("A tiny gap is expected: the ground changes runs per ball by about 0.02 on average. It is kept because it removes a "
      "systematic unfairness (a batter at Chennai or Lucknow is not blamed for the pitch).")

# 5c) THE "NO SKILL LEAKS IN" CHECK. A good engine describes the SITUATION, so across a whole season, players should not all
# have the same average gap. If the model had absorbed player skill, every player's average gap would collapse to ~0.
# Here: spread (standard deviation) of each batter's average runs-above-expected per ball, for batters with 500+ balls.
g = ball[rows_bat].assign(gap=lambda d: d.y_bat_runs - d.exp_bat_runs).groupby("striker_id").gap.agg(["size", "mean"]).query("size >= 500")
noise = np.sqrt((ball.loc[rows_bat, "y_bat_runs"] - ball.loc[rows_bat, "exp_bat_runs"]).var() / g["size"]).mean()
print(f"\nBatters with 500+ balls: {len(g)}. Spread of their average gap: {g['mean'].std():.3f} runs/ball. "
      f"Spread expected from pure luck alone: {noise:.3f}. Real skill differences show up as spread BEYOND luck.")

Forward test (train up to 2025, predict 2026):


,2026 RMSE: model,2026 RMSE: league avg (from <=2025),avg predicted,avg actual
batter runs,1.7933,1.8273,1.4648,1.5052
runs conceded,1.7669,1.7972,1.4561,1.4956


If 'avg predicted' sits a little below 'avg actual', that is the 2026 scoring rise the model could not see in the past; grouped CV above includes 2026 in training, which is why the final engine uses all seasons.


,"RMSE, batter runs"
with ground features,1.7099
without ground features,1.7106


A tiny gap is expected: the ground changes runs per ball by about 0.02 on average. It is kept because it removes a systematic unfairness (a batter at Chennai or Lucknow is not blamed for the pitch).

Batters with 500+ balls: 122. Spread of their average gap: 0.114 runs/ball. Spread expected from pure luck alone: 0.056. Real skill differences show up as spread BEYOND luck.


### Cell 5 · Forward test, ground features, skill check

**Procedure**
1. Trained on matches up to 2025 only and predicted 2026.
2. Re-ran the batter-runs model without the four ground features and compared.
3. Measured how widely batters' average gaps (actual minus expected) are spread across the 122 batters with 500+ balls, against the spread luck alone would produce.

**Observations**
- 2026 forward test: the model beats the league average (RMSE 1.793 against 1.827). It predicts 1.465 runs a ball against an actual 1.505. The shortfall is the 2026 scoring rise the model could not see in older data. The final engine trains on all seasons.
- Ground features change accuracy by almost nothing (1.7100 against 1.7104). They stay for fairness, not accuracy, so a batter at Chennai or Lucknow is not blamed for the pitch.
- Skill check: batters' gaps spread by 0.114 runs a ball, against 0.056 from luck alone. The excess is real skill (about 0.10), which shows the engine has not absorbed player skill.

In [6]:
# ─────────────────────────────────────────────────────────────────────
# CELL 6 · WHAT THE ENGINE LEARNED (and does it make cricket sense?)
# ─────────────────────────────────────────────────────────────────────
# 6a) FEATURE IMPORTANCE: which inputs matter most. TECHNICAL: "gain" = how much total error reduction a feature provided
# across all the trees. Fitted once on all data (this model is only for explanation, not for the ratings).
m_exp, _ = fit_predict("y_bat_runs", "regression", np.arange(len(ball)), np.arange(10), list(X.columns), R_BAT)
imp = pd.Series(m_exp.feature_importance("gain"), index=X.columns).sort_values(ascending=False)
display((imp / imp.sum()).head(15).round(3).to_frame("share of total gain"))

# 6b) COMMON SENSE TABLES. If the engine is right, it should reproduce what any cricketer knows.
# Expected runs per ball by over, batting first vs chasing:
tab = ball[ball.faced].groupby(["is_chase", "over"]).exp_bat_runs.mean().unstack(0).rename(columns={False: "batting first", True: "chasing"})
tab["actual (all)"] = ball[ball.faced].groupby("over").y_bat_runs.mean()
display(tab.round(3).T)
print("Expect: a slow first over, a build-up through the powerplay (overs 1-6), a drop when fielding restrictions end at over 7, "
      "a steady climb through the middle overs, and a surge in the last four overs.")

# Wickets: a wicket on the previous balls (new batter) should raise the risk; death overs with a steep target should raise it too.
ball["rrr_band"] = pd.cut(ball.rrr, [0, 6, 8, 10, 12, 15, 40], labels=["<=6", "6-8", "8-10", "10-12", "12-15", "15+"])
w = ball[ball.faced & (ball.innings == 2)].groupby("rrr_band").agg(balls=("p_bat_out", "size"), expected_out_rate=("p_bat_out", "mean"), actual_out_rate=("y_bat_out", "mean"),
                                                                   expected_runs=("exp_bat_runs", "mean"), actual_runs=("y_bat_runs", "mean"))
print("\nChases, by required run rate:"); display(w.round(4))
nb = ball[ball.faced].groupby(["new_batter"]).agg(balls=("p_bat_out", "size"), expected_out_rate=("p_bat_out", "mean"), actual_out_rate=("y_bat_out", "mean"),
                                                 expected_runs=("exp_bat_runs", "mean"), actual_runs=("y_bat_runs", "mean"))
print("\nNew batter (<5 balls faced) vs settled:"); display(nb.round(4))

,share of total gain
bat_balls_before,0.1960
score_before,0.1280
pship_runs_before,0.1160
crr,0.0460
balls_left,0.0430
entry_wkts,0.0370
venue_boundary_effect,0.0270
elo_diff,0.0270
wkts_before,0.0250
venue_par_effect,0.0230


over,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
is_chase,,,,,,,,,,,,,,,,,,,,
batting first,1.0430,1.2720,1.3940,1.4280,1.4450,1.4360,1.2550,1.2830,1.2950,1.3120,1.3300,1.3470,1.3740,1.3890,1.4150,1.4920,1.5620,1.6170,1.6850,1.8180
chasing,1.0870,1.2980,1.4180,1.4590,1.4540,1.4380,1.2670,1.2940,1.3130,1.3160,1.3270,1.3540,1.3770,1.4030,1.4330,1.4460,1.4900,1.5380,1.5970,1.6840
actual (all),1.0700,1.2530,1.4260,1.4400,1.4830,1.4670,1.2070,1.2610,1.3120,1.3150,1.3190,1.3530,1.3670,1.3980,1.4260,1.4630,1.5250,1.5940,1.6450,1.8020


Expect: a slow first over, a build-up through the powerplay (overs 1-6), a drop when fielding restrictions end at over 7, a steady climb through the middle overs, and a surge in the last four overs.

Chases, by required run rate:


,balls,expected_out_rate,actual_out_rate,expected_runs,actual_runs
rrr_band,,,,,
<=6,11472,0.0504,0.0476,1.4082,1.4030
6-8,16012,0.0456,0.0445,1.2860,1.2666
8-10,23130,0.0477,0.0465,1.3181,1.3173
10-12,19014,0.0543,0.0543,1.4148,1.4233
12-15,11448,0.0658,0.0692,1.4797,1.4800
15+,10536,0.0905,0.0910,1.4882,1.4809



New batter (<5 balls faced) vs settled:


,balls,expected_out_rate,actual_out_rate,expected_runs,actual_runs
new_batter,,,,,
False,133871,0.0572,0.0567,1.5223,1.5238
True,59380,0.0500,0.0502,1.1095,1.1032


### Cell 6 · What the engine learned

**Procedure**
1. Ranked inputs by "gain" (total error reduction across all trees).
2. Printed expected runs per ball by over for batting first and chasing, and compared them with actual.
3. Compared expected and actual dismissal and run rates by required run rate in chases, and for new vs settled batters.

**Observations**
- The biggest drivers are the batter's balls faced so far (18.5%), partnership runs (12.3%), score so far (12.3%) and current run rate (4.9%). Ground features each contribute about 2 to 3%.
- The over-by-over shape matches cricket: a slow over 1 (about 1.05), a build-up through the powerplay to about 1.45, a drop to about 1.25 at over 7 when fielding restrictions end, a steady climb, then a surge in the death.
- In chases, the dismissal rate rises from about 5% (required rate up to 6) to 9% (above 15 an over), and the model tracks it closely.
- New batters (under 5 balls) score less per ball (1.11 against 1.52) and are dismissed no more often than settled ones (5.0% against 5.7%). That is what the data shows, and the engine reproduces it.

In [7]:
# ─────────────────────────────────────────────────────────────────────
# CELL 7 · IS THE ENGINE FAIR ACROSS CONTEXTS? (the gap should average ~0 everywhere)
# Purpose: if the engine is systematically off in some context (a phase, a tier, a pitch type, a year), then players who
#          play mostly there would be unfairly praised or blamed. We check this before building any rating.
# ─────────────────────────────────────────────────────────────────────
# GAP = actual - expected. Averages near 0 mean fair. The "luck band" is 2 standard errors: a gap inside it cannot be told from chance.
def fairness(by, target, exp, rows):
    d = ball[rows].assign(gap=ball[target] - ball[exp])
    t = d.groupby(by).gap.agg(balls="size", avg_gap="mean", sd="std")
    t["luck_band(+-)"] = 2 * t.sd / np.sqrt(t.balls)
    t["inside_luck_band"] = t.avg_gap.abs() <= t["luck_band(+-)"]
    return t.drop(columns="sd")
rb = ball.faced
print("BATTER RUNS: average gap (actual - expected, runs per ball)")
# Ground check: split balls into thirds by the model's own (leave-one-out) ground effect. We do NOT use the readable pitch_type tag
# here: the tag was built from all matches including these very balls, so a "flat" ground is flat partly BECAUSE its matches scored
# high, which would make this check circular.
ball["ground_third"] = pd.qcut(ball.venue_par_effect, 3, labels=["lowest-scoring grounds", "middle", "highest-scoring grounds"], duplicates="drop")
for by in ["phase", "tier", "year", "ground_third"]:
    display(fairness(by, "y_bat_runs", "exp_bat_runs", rb).round(4))
print("\nRUNS CONCEDED: average gap")
for by in ["phase", "pace_or_spin", "year"]:
    display(fairness(by, "y_runs_conc", "exp_runs_conc", np.ones(len(ball), bool)).round(4))
print("\nNote: the luck band treats balls as independent, but balls in one match are linked, so the true band is somewhat wider. "
      "Gaps of a few hundredths of a run per ball (a couple of runs per innings) are therefore not worrying.")
ball = ball.drop(columns=["ground_third"])

# 7b) The sample sizes that matter later. How many balls will each player have under pressure? Here: a rough preview using the
# raw situation "chasing, required rate above 10 per over" as a stand-in until the Leverage Index exists (notebook 05).
tough = ball[ball.faced & (ball.innings == 2) & (ball.rrr > 10)]
print(f"\nPreview: batters with 60+ balls in 'chasing above 10 an over': {(tough.groupby('striker_id').size() >= 60).sum()} "
      f"(of {ball[ball.faced].striker_id.nunique()} batters). Pressure samples are small, which is why shrinkage will be essential.")

BATTER RUNS: average gap (actual - expected, runs per ball)


,balls,avg_gap,luck_band(+-),inside_luck_band
phase,,,,
death,42003,0.0029,0.0178,True
middle,89552,-0.0094,0.0107,True
powerplay,61696,0.0090,0.0143,True


,balls,avg_gap,luck_band(+-),inside_luck_band
tier,,,,
A-team / emerging,6735,-0.0235,0.0399,True
IPL,84121,0.0004,0.0121,True
T20I,102395,-0.0004,0.0104,True


,balls,avg_gap,luck_band(+-),inside_luck_band
year,,,,
2022,40851,-0.0156,0.0163,True
2023,31813,-0.0059,0.0190,True
2024,43070,-0.0001,0.0164,True
2025,45824,0.0050,0.0161,True
2026,31693,0.0137,0.0201,True


,balls,avg_gap,luck_band(+-),inside_luck_band
ground_third,,,,
lowest-scoring grounds,64686,-0.0083,0.0127,True
middle,64264,0.0128,0.0138,True
highest-scoring grounds,64301,-0.0070,0.0138,True



RUNS CONCEDED: average gap


,balls,avg_gap,luck_band(+-),inside_luck_band
phase,,,,
death,44454,-0.0033,0.0170,True
middle,92499,-0.0093,0.0104,True
powerplay,64008,0.0046,0.0139,True


,balls,avg_gap,luck_band(+-),inside_luck_band
pace_or_spin,,,,
Pace,123859,-0.0007,0.0098,True
Spin,77102,-0.0080,0.0116,True


,balls,avg_gap,luck_band(+-),inside_luck_band
year,,,,
2022,42465,-0.0148,0.0158,True
2023,32988,-0.0095,0.0184,True
2024,44790,-0.0014,0.0159,True
2025,47636,0.0034,0.0156,True
2026,33082,0.0039,0.0194,True



Note: the luck band treats balls as independent, but balls in one match are linked, so the true band is somewhat wider. Gaps of a few hundredths of a run per ball (a couple of runs per innings) are therefore not worrying.

Preview: batters with 60+ balls in 'chasing above 10 an over': 189 (of 888 batters). Pressure samples are small, which is why shrinkage will be essential.


### Cell 7 · Fairness across contexts

**Procedure**
1. Computed the average gap (actual minus expected) by phase, tier, year and ground type for batter runs, and by phase, pace or spin, and year for runs conceded.
2. Compared each average with a "luck band" of two standard errors.
3. Split grounds into thirds by the model's own leave-one-out ground effect. The readable pitch tags were not used, because they were built from the same matches and the check would be circular.

**Observations**
- Every gap is inside its luck band, except one: runs conceded in 2022 (-0.017 runs a ball, about 2 runs an innings). The model slightly over-expects runs in the earliest season.
- The luck band treats balls as independent, but balls within a match are linked, so the true band is somewhat wider. Gaps of a few hundredths of a run are not worrying.
- Nobody is unfairly helped or hurt by phase, tier or ground type. This is what lets us credit the remaining gap to the player.
- Pressure samples are small: only 190 of 888 batters have 60+ balls chasing above 10 an over, which is why shrinkage will be essential.

In [8]:
# ─────────────────────────────────────────────────────────────────────
# CELL 8 · A FIRST LOOK AT PLAYERS: runs above expected (RAE) and runs saved. Then SAVE the engine.
# Purpose: a sanity check with real names, before pressure enters. We rank players on ALL balls (no pressure yet) and see
#          whether the leaders match what an IPL follower would expect. This is NOT the rating; it is a smoke test.
# ─────────────────────────────────────────────────────────────────────
name = players.set_index("player_id").display_name
d = ball[ball.faced].assign(gap=lambda t: t.y_bat_runs - t.exp_bat_runs)
bat = d.groupby("striker_id").agg(balls=("gap", "size"), runs=("y_bat_runs", "sum"), expected=("exp_bat_runs", "sum"), gap=("gap", "sum")).query("balls >= 600")
bat["actual_sr"] = 100 * bat.runs / bat.balls
bat["rae_per_100"] = 100 * bat.gap / bat.balls                       # runs above expected, per 100 balls faced
# TRUE STRIKE RATE (first version): the league-average strike rate plus the player's per-100 gap = "strike rate after
# the situation, ground and opposition are held equal". It removes the benefit of an easy slot (e.g. batting at the death on a flat pitch).
lg_sr = 100 * d.y_bat_runs.mean()
bat["true_sr"] = lg_sr + bat.rae_per_100
bat["luck_band(+-)"] = 2 * 100 * d.groupby("striker_id").gap.std().reindex(bat.index) / np.sqrt(bat.balls)
bat.insert(0, "player", bat.index.map(name))
print(f"League average strike rate on this sample: {lg_sr:.1f}")
print("\nTop 12 batters by runs above expected per 100 balls (600+ balls):"); display(bat.sort_values("rae_per_100", ascending=False).head(12).round(1))
print("\nBottom 8:"); display(bat.sort_values("rae_per_100").head(8).round(1))

dd = ball.assign(gap=lambda t: t.exp_runs_conc - t.y_runs_conc)       # positive = runs SAVED relative to expectation
bow = dd.groupby("bowler_id").agg(balls=("gap", "size"), conceded=("y_runs_conc", "sum"), expected=("exp_runs_conc", "sum"), saved=("gap", "sum"),
                                  wkts=("y_bowl_wkt", "sum"), exp_wkts=("p_bowl_wkt", "sum")).query("balls >= 400")
lg_econ = 6 * dd.y_runs_conc.mean()
bow["actual_econ"] = 6 * bow.conceded / bow.balls
bow["saved_per_24balls"] = 24 * bow.saved / bow.balls              # runs saved per 4 overs
bow["true_econ"] = lg_econ - 6 * bow.saved / bow.balls              # league economy minus the runs saved per over (TRUE ECONOMY, first version)
bow["wkts_above_exp_per_24"] = 24 * (bow.wkts - bow.exp_wkts) / bow.balls
bow.insert(0, "player", bow.index.map(name))
print(f"\nLeague average economy: {lg_econ:.2f}")
print("\nTop 12 bowlers by runs saved per 4 overs (400+ balls):"); display(bow.sort_values("saved_per_24balls", ascending=False).head(12)[["player", "balls", "actual_econ", "true_econ", "saved_per_24balls", "wkts_above_exp_per_24"]].round(2))
print("\nBottom 8:"); display(bow.sort_values("saved_per_24balls").head(8)[["player", "balls", "actual_econ", "true_econ", "saved_per_24balls", "wkts_above_exp_per_24"]].round(2))

# 8b) SAVE. (1) The expectations on every ball, for notebook 05 onwards. (2) The final engines, trained on ALL matches, for the website
# and for scoring any "what if" situation later. The ratings use the out-of-fold numbers above, never these final engines.
ball = ball.drop(columns=["rrr_band"], errors="ignore")
print("\nSaved ball_master:", ball.shape, save_table(ball, "ball_master"))
MODEL_DIR = OUT_DIR / "models"; MODEL_DIR.mkdir(exist_ok=True)
allidx = np.arange(len(ball))
for tgt, obj, rounds in [("y_bat_runs", "regression", R_BAT), ("y_runs_conc", "regression", R_CONC),
                         ("y_bat_out", "binary", int(np.mean(best_out) * 1.1)), ("y_bowl_wkt", "binary", int(np.mean(best_wkt) * 1.1))]:
    mdl, _ = fit_predict(tgt, obj, allidx, allidx[:5], list(X.columns), rounds)
    mdl.save_model(str(MODEL_DIR / f"{tgt}.txt"))
pd.Series(list(X.columns)).to_csv(MODEL_DIR / "features.csv", index=False, header=False)
print("Saved 4 final engines in", MODEL_DIR)

League average strike rate on this sample: 139.5

Top 12 batters by runs above expected per 100 balls (600+ balls):


,player,balls,runs,expected,gap,actual_sr,rae_per_100,true_sr,luck_band(+-)
striker_id,,,,,,,,,
50284,Vaibhav Sooryavanshi,676,"1,509.0000","1,114.7000",394.3000,223.2000,58.3000,197.8000,17.9000
4847,Finn Allen,834,"1,455.0000","1,192.5000",262.5000,174.5000,31.5000,170.9000,14.6000
6281,Abhishek Sharma,2317,"4,332.0000","3,681.4000",650.6000,187.0000,28.1000,167.5000,8.5000
1318,Andre Russell,733,"1,254.0000","1,062.1000",191.9000,171.1000,26.2000,165.6000,15.9000
6396,Tim David,1226,"2,178.0000","1,861.3000",316.7000,177.7000,25.8000,165.3000,11.4000
5039,Rajat Patidar,918,"1,541.0000","1,357.0000",184.0000,167.9000,20.0000,159.5000,12.7000
5553,Phil Salt,1643,"2,782.0000","2,473.5000",308.5000,169.3000,18.8000,158.2000,9.3000
838,Glenn Maxwell,958,"1,542.0000","1,362.7000",179.3000,161.0000,18.7000,158.2000,12.2000
1699,Suryakumar Yadav,2759,"4,588.0000","4,080.3000",507.7000,166.3000,18.4000,157.9000,7.1000



Bottom 8:


,player,balls,runs,expected,gap,actual_sr,rae_per_100,true_sr,luck_band(+-)
striker_id,,,,,,,,,
6448,Ibrahim Zadran,1126,"1,319.0000","1,504.5000",-185.5000,117.1000,-16.5000,123.0000,8.6000
152,Ravindra Jadeja,958,"1,294.0000","1,443.7000",-149.7000,135.1000,-15.6000,123.8000,9.6000
109,Wriddhiman Saha,661,824.0000,917.5000,-93.5000,124.7000,-14.1000,125.3000,12.1000
142,Shikhar Dhawan,757,985.0000,"1,079.2000",-94.2000,130.1000,-12.4000,127.0000,11.7000
4219,Krunal Pandya,656,839.0000,917.6000,-78.6000,127.9000,-12.0000,127.5000,12.2000
3024,Mohammad Rizwan,1152,"1,467.0000","1,589.9000",-122.9000,127.3000,-10.7000,128.8000,8.5000
1941,Devon Conway,1486,"1,968.0000","2,126.2000",-158.2000,132.4000,-10.6000,128.8000,8.4000
4812,Harry Tector,828,"1,015.0000","1,097.2000",-82.2000,122.6000,-9.9000,129.5000,10.0000



League average economy: 8.35

Top 12 bowlers by runs saved per 4 overs (400+ balls):


,player,balls,actual_econ,true_econ,saved_per_24balls,wkts_above_exp_per_24
bowler_id,,,,,,
2933,Jasprit Bumrah,1780,6.9700,6.2800,8.2600,-0.1300
781,Sunil Narine,1553,6.7700,6.9600,5.5400,-0.0800
7920,Nathan Ellis,959,7.8800,7.0500,5.2000,-0.0100
34485,Sufiyan Muqeem,525,5.9200,7.1600,4.7300,0.4400
4373,Mustafizur Rahman,1646,7.4700,7.2900,4.2200,-0.2200
6523,Blessing Muzarabani,846,7.3000,7.3500,4.0000,0.0100
8661,Matheesha Pathirana,1279,7.8300,7.3700,3.9000,0.1600
5979,Abrar Ahmed,776,6.5800,7.4700,3.5100,0.1100
39657,Eshan Malinga,989,8.4500,7.4900,3.4200,-0.0100



Bottom 8:


,player,balls,actual_econ,true_econ,saved_per_24balls,wkts_above_exp_per_24
bowler_id,,,,,,
7472,Shahbaz Ahmed,554,10.1000,10.0600,-6.8500,-0.3200
3605,Aiden Markram,408,9.8200,9.8400,-5.9800,-0.3300
1318,Andre Russell,792,10.4200,9.6900,-5.3500,0.5200
4755,Shivam Dube,491,9.4900,9.6100,-5.0400,0.1100
20005,Arshad Khan,413,10.0700,9.4800,-4.5100,-0.0600
8380,Mohammad Rishad Hossain,1033,8.3600,9.4100,-4.2400,0.1300
4391,Liam Livingstone,619,9.2800,9.4000,-4.2200,0.0800
4840,Odean Smith,414,9.7500,9.3600,-4.0400,0.1300



Saved ball_master: (200961, 108) parquet
Saved 4 final engines in interim\models


### Cell 8 · First look at players, and saving the engine

**Procedure**
1. For each batter with 600+ balls, computed runs above expected per 100 balls and a first version of True Strike Rate (league strike rate 139.5 plus that gap).
2. For each bowler with 400+ balls, computed runs saved per 4 overs and a first version of True Economy (league economy 8.30 minus the runs saved per over).
3. Saved the expected values on every ball, and trained four final engines on all matches for later use.

**Observations**
- This is a smoke test on all balls with no pressure yet, not the rating.
- Top batters are Vaibhav Sooryavanshi (about +57 runs per 100 balls), Finn Allen, Abhishek Sharma, Andre Russell, Tim David, Rajat Patidar, Glenn Maxwell and Phil Salt. The bottom includes Zadran, Jadeja, Saha, Dhawan, Rizwan and Tector, who are anchors and slow starters. The pattern matches what a follower of the league would expect.
- Top bowlers are Jasprit Bumrah (actual economy 6.89, true economy about 6.2, because he bowls the hardest overs), Sunil Narine, Nathan Ellis, Matheesha Pathirana, Mustafizur Rahman and Abrar Ahmed. The bottom includes Shahbaz Ahmed, Markram, Russell and Shivam Dube, who are used for part-time overs in hard situations.
- Small samples shift rankings: Sooryavanshi has 676 balls and a luck band of about ±18 runs per 100, so his place needs shrinkage before it means anything. Players near the thresholds move easily too.
- Saved: expectations on every ball (ball_master now has 108 columns) and four final engines in `interim/models`.